# ReAct (Reason + Act) — a step-by-step, runnable teaching notebook

This notebook builds a **real ReAct agent** from the ground up, one operation at a time, driving a genuine small instruction-tuned LLM (`Qwen/Qwen2.5-1.5B-Instruct`) through a real **Thought → Action → Observation** loop against real Python tools. It is the executable companion to the chapter and to `react_agent.py` — every function used here lives in that module, imported so the notebook and the module can never drift apart.

Nothing about the model's output is mocked. The traces below are what the model actually generates (greedy / temperature 0, so they are reproducible). By the end you will have **seen**, with a real model and real tools:

1. why an LLM alone gets multi-step questions **confidently wrong**;
2. two **real tools** — a safe calculator (AST-walked, not `eval`) and a local `wiki` lookup;
3. the **ReAct prompt grammar** — the Thought/Action/Observation format with a one-shot example;
4. the **stop condition** that halts the model *before* it can hallucinate an observation;
5. **robust parsing** of messy model text into a structured action;
6. the **full loop** — reason, act, observe, repeat — on a numeric and a multi-hop question;
7. a **head-to-head**: ReAct (with tools) vs reason-only (no tools) on real multi-step questions.

The first run downloads the model (a few hundred MB) and caches it; every run after is offline and reproducible. It runs on CPU, Apple MPS, or CUDA — whatever you have.

## Step 0 — Setup and version banner

We import the real pieces from the chapter module (so this notebook uses the *exact same code* the chapter and figures use) and print the library + model versions and the device the results were produced on. `pick_device()` chooses `cuda → mps → cpu` without assuming a GPU.

In [1]:
import torch
import transformers

from react_agent import (
    calculator, wiki, KNOWLEDGE_BASE,
    SYSTEM_PROMPT,
    LanguageModel, pick_device,
    parse_action, _normalise_finish,
    dispatch, run_react, run_direct,
    compare_react_vs_direct, EVAL_SET,
)

print(f'torch {torch.__version__} | transformers {transformers.__version__} '
      f'| device {pick_device()}')

torch 2.12.0 | transformers 5.10.2 | device mps


## Step 1 — The problem: an LLM alone guesses, and guesses wrong

Before any agent machinery, feel the gap. We ask the model — with **no tools** — a question that needs one exact multiplication and one addition. A capable-sounding model will produce a confident number. Watch whether it is *right*.

This is the reason-only baseline (`run_direct`): one shot, no reasoning trace, no tools — exactly what an LLM does on its own.

In [2]:
llm = LanguageModel()   # loads the real model (downloads+caches on first run)
print(f'loaded {llm.model_id} on {llm.device}')

q = 'What is 481 multiplied by 32, then plus 19?'
direct_answer = run_direct(llm, q)
print('question :', q)
print('LLM alone:', direct_answer, '   (correct answer is 15411)')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

loaded Qwen/Qwen2.5-1.5B-Instruct on mps


question : What is 481 multiplied by 32, then plus 19?
LLM alone: 15605    (correct answer is 15411)


The model answers fluently but the arithmetic is off — it is doing multi-digit multiplication *in its head*, token by token, and small models are unreliable at that. It cannot check itself. This is the felt inadequacy ReAct removes: give the model a **tool** and let it *act*, then *read the real result*.

## Step 2 — Real tool #1: a safe calculator (AST-walked, not `eval`)

The first tool is a real calculator. Crucially it is **not** `eval(expr)` — that would let a model (or a prompt-injection) run arbitrary code. Instead we parse the expression to an abstract syntax tree and walk it, permitting *only* numbers and a whitelist of arithmetic operators. Anything else raises. Safe tool design is part of the lesson.

In [3]:
print(calculator('481 * 32 + 19'))     # the real answer the LLM missed
print(calculator('17 ** 3 - 200'))     # powers work too
print(calculator('(1287 - 998) * 6'))  # parentheses respected

# and it refuses anything that is not pure arithmetic (this is what makes it SAFE):
try:
    calculator('__import__("os").system("echo hi")')
except Exception as e:
    print('rejected unsafe input:', type(e).__name__, '-', e)

15411
4713
1734
rejected unsafe input: ToolError - expression contains a disallowed operation


## Step 3 — Real tool #2: a `wiki` lookup against a local knowledge base

The second tool returns real *facts* the model may not know — a stand-in for a web/Wikipedia search, but offline and deterministic so the notebook reproduces exactly. The agent must **read** the returned text to answer multi-hop questions. Note the deliberate miss case: a real tool sometimes returns nothing useful, and the agent has to cope.

In [4]:
print('KB topics:', list(KNOWLEDGE_BASE)[:4], '...')
print()
print(wiki('Eiffel Tower'))
print()
print(wiki('the moon landing'))
print()
print(wiki('quantum chromodynamics'))   # a real miss — the agent must handle this

KB topics: ['eiffel tower', 'great pyramid of giza', 'moon landing', 'python programming language'] ...

The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889 as the entrance arch to the 1889 World's Fair and stands 330 metres tall.

Apollo 11 landed the first humans on the Moon on 20 July 1969. Neil Armstrong and Buzz Aldrin walked on the surface while Michael Collins orbited above.

No knowledge-base entry found for 'quantum chromodynamics'.


## Step 4 — The ReAct prompt: the Thought / Action / Observation grammar

ReAct is, mechanically, a **prompt format plus a loop**. The system prompt tells the model to emit exactly one `Thought:` line then one `Action: tool[input]` line and then *stop*, and it includes a single worked example (one-shot) so a small model reliably copies the shape. The tools it may call and the special `finish[...]` action are declared here.

In [5]:
print(SYSTEM_PROMPT)

You solve questions by reasoning and using tools, one step at a time.

At each step output EXACTLY one Thought line then one Action line, then STOP and wait:
Thought: <your reasoning about what to do next>
Action: <tool>[<input>]

Available tools:
- calculator[expression]   evaluates arithmetic, e.g. Action: calculator[481 * 32 + 19]
- wiki[query]              looks up a fact, e.g. Action: wiki[Eiffel Tower]
- finish[answer]           gives the FINAL answer, e.g. Action: finish[15411]

Rules:
- Never write an "Observation:" line yourself — the system supplies real observations.
- Put the fully computed final value inside finish[...], not an expression.

Example:
Question: What is 6 times 7, plus 3?
Thought: I should compute 6 * 7 + 3 with the calculator.
Action: calculator[6 * 7 + 3]
Observation: 45
Thought: The calculator returned 45, which is the final answer.
Action: finish[45]

Now solve the new question.


## Step 5 — One raw generation: the model proposes a Thought + Action

Let's run a *single* generation step by hand. We feed the system prompt + the question and let the model produce text. The key move: generation is **halted at the first `Observation:`** (a stop string), so the model proposes an action but does **not** get to invent the tool's result. That 'stop before it fakes the observation' is the heart of ReAct.

In [6]:
raw = llm.generate(SYSTEM_PROMPT, f'Question: {q}\n')
print(raw)

Thought: To find the result of multiplying 481 by 32 and adding 19, I need to use the calculator function on these numbers.
Action: calculator[481 * 32 + 19]
Thought: After calculating, the result is 15411.
Action: finish[15411]


## Step 6 — Parsing messy model text into a structured Action

Real models are messy: extra lines, chatter, occasional bracket typos. `parse_action` finds the **first** well-formed `Action: tool[arg]` with a regex and throws away everything after it — so a chatty model can never advance the loop by more than one real step. It returns the trimmed text and a typed `Action(tool, arg)` (or `None` if the model failed to emit a parseable action — a real failure the loop must survive).

In [7]:
_, action = parse_action(raw)   # first return value is the trimmed text; we only need the action
print('parsed action ->', action)
print('tool:', action.tool, '| arg:', repr(action.arg))

parsed action -> Action(tool='calculator', arg='481 * 32 + 19')
tool: calculator | arg: '481 * 32 + 19'


## Step 7 — Dispatch: run the real tool, get the REAL observation

`dispatch` looks the tool up in the registry and calls it with the parsed argument, returning its real result string. Unknown tools and tool errors become *observations* (not crashes), so the agent can read the problem and recover — how a robust ReAct loop behaves in the wild.

In [8]:
observation = dispatch(action)
print('Observation:', observation)

Observation: 15411


## Step 8 — The full loop on a numeric question

Now assemble Steps 5–7 into the loop. `run_react` grows a scratchpad one real step at a time: append the model's Thought + Action, splice in the **real** Observation, ask again with the enlarged context, and stop on `finish[...]`, a parse failure, or the step budget (the guard against infinite loops). Here is the complete real trace for the numeric question the LLM missed in Step 1.

In [9]:
result = run_react(llm, q)
print(result.transcript())
print()
print(f'stop_reason={result.stop_reason} | steps={result.num_steps} | '
      f'tool_calls={result.num_tool_calls} | answer={result.answer}')

Question: What is 481 multiplied by 32, then plus 19?
Thought: To find the result of multiplying 481 by 32 and adding 19, I need to use the calculator function on these numbers.
Action: calculator[481 * 32 + 19]
Observation: 15411
Thought: The calculation has been performed successfully. Now, I will present the final answer.
Action: finish[15411]
Answer: 15411

stop_reason=finish | steps=2 | tool_calls=1 | answer=15411


The same model that guessed wrong in Step 1 now gets it **exactly right** — because it *acted* (called the calculator) and *read the real observation* instead of trusting its head. That is the entire ReAct thesis in one before/after.

## Step 9 — A multi-hop trace: `wiki` THEN `calculator`

The real power shows on questions that need **two different tools in sequence**: look a fact up, then compute with it. The agent has to decide, at each step, *which* tool to use next — driven purely by what it read in the previous observation. No fixed plan; it reacts.

In [10]:
mh = run_react(llm, 'In what year was the Eiffel Tower completed, and what is that year plus 100?')
print(mh.transcript())

Question: In what year was the Eiffel Tower completed, and what is that year plus 100?
Thought: To find out when the Eiffel Tower was completed, I need to look it up in a reliable source or tool. Let's use the wiki[query] action for this purpose.
Action: wiki[Eiffel Tower completion date]
Observation: The Eiffel Tower is a wrought-iron lattice tower in Paris, France. It was completed in 1889 as the entrance arch to the 1889 World's Fair and stands 330 metres tall.
Thought: From the information provided, we can see that the Eiffel Tower was completed in 1889. Now, let's calculate the year 100 years after 1889.
Action: calculator[1889 + 100]
Observation: 1989
Thought: The calculation shows that 100 years after 1889 is 1989. This is the final answer.
Action: finish[1989]
Answer: 1989


Read the flow: `wiki` returns the real sentence containing **1889**; the model reads it, reasons '1889 + 100', calls the calculator, gets **1989**, and finishes. The fact came from the tool, not from the model's memory — which is precisely how ReAct reduces hallucination.

## Step 10 — A real robustness detail: normalising `finish`

Small models often emit `finish[1889 + 100]` — the *expression* rather than the *value*. Rather than score that wrong, `_normalise_finish` evaluates a purely-numeric finish argument through the same safe calculator, reflecting the model's obvious intent. Non-numeric answers pass through untouched. This kind of defensive normalisation is 80% of making a real agent work.

In [11]:
print(_normalise_finish('1889 + 100'))   # numeric expression -> evaluated
print(_normalise_finish('15411'))        # already a number -> unchanged
print(_normalise_finish('Ada Lovelace')) # a phrase -> passed through

1989
15411
Ada Lovelace


## Step 11 — ReAct vs reason-only: the head-to-head on real questions

The claim of the ReAct paper is that interleaving reasoning with *acting on real observations* beats reasoning alone. We test it honestly: `compare_react_vs_direct` runs the **same** real multi-step questions two ways — the full ReAct loop vs a single 'answer directly, no tools' prompt — and scores exact match. Every generation is greedy, so this table reproduces exactly.

In [12]:
rows = compare_react_vs_direct(llm, EVAL_SET)
print(f"{'gold':>8} | {'ReAct':>8} {'ok':>3} {'steps':>5} | {'direct':>12} {'ok':>3}")
print('-' * 54)
for r in rows:
    print(f'{r.gold:>8} | {str(r.react_answer):>8} '
          f"{'Y' if r.react_correct else 'N':>3} {r.react_steps:>5} | "
          f"{str(r.direct_answer)[:12]:>12} {'Y' if r.direct_correct else 'N':>3}")

    gold |    ReAct  ok steps |       direct  ok
------------------------------------------------------
   15411 |    15411   Y     2 |        15605   N
    1734 |     1734   Y     3 |           30   N
    1989 |     1989   Y     3 |   1889; 1989   Y
     438 |      438   Y     2 |          438   Y
    8000 |     8000   Y     2 |         8000   Y
    4713 |     4713   Y     3 |          491   N


In [13]:
react_acc = sum(r.react_correct for r in rows) / len(rows)
direct_acc = sum(r.direct_correct for r in rows) / len(rows)
print(f'ReAct accuracy  : {react_acc:.0%} ({sum(r.react_correct for r in rows)}/{len(rows)})')
print(f'Direct accuracy : {direct_acc:.0%} ({sum(r.direct_correct for r in rows)}/{len(rows)})')

ReAct accuracy  : 100% (6/6)
Direct accuracy : 50% (3/6)


On this real set the reason-only model is right about half the time — it nails the questions where the arithmetic is easy and blows the ones needing exact multi-digit computation or a looked-up fact. ReAct, by grounding every step in a real tool result, gets them **all**. The gap *is* the value of acting.

## Step 12 — Watching a failure mode: the step budget stops a runaway loop

ReAct loops can misbehave — the classic failure is a model that never calls `finish` and loops forever. The `max_steps` budget is the real stop condition that prevents that. We force a tiny budget on a harder question and watch the loop terminate cleanly with `stop_reason='step_budget'` instead of hanging. (In production you'd surface this as 'I couldn't solve it in N steps', not a crash.)

In [14]:
capped = run_react(llm, 'What is 17 to the power of 3, minus 200?', max_steps=1)
print(capped.transcript())
print()
print('stop_reason:', capped.stop_reason, '| answer:', capped.answer,
      '  <- one step was not enough; the budget stopped it safely')

Question: What is 17 to the power of 3, minus 200?
Thought: To find the result, I need to calculate 17^3 first and then subtract 200 from it. This can be done using the calculator tool.
Action: calculator[17^3 - 200]
Observation: Error: expression contains a disallowed operation

stop_reason: step_budget | answer: None   <- one step was not enough; the budget stopped it safely


## Step 13 — The figures on the chapter page come from exactly this run

Every figure in the chapter is generated from the same real module you just ran — no hand-typed numbers. The trace figure is a real solved trace (Step 9); the accuracy bars and the per-question grid are the real comparison table (Step 11). You can regenerate them yourself:

```bash
python "../../tools/make_figures_02.py"   # writes agentic02_*.png into ../../images/
```

That closes the loop between the page, the notebook, and the module: one real agent, demonstrated three ways, always in agreement.

In [15]:
# Confirm the numbers behind the chapter's comparison figure, from THIS run:
print('ReAct correct :', sum(r.react_correct for r in rows), '/', len(rows))
print('Direct correct:', sum(r.direct_correct for r in rows), '/', len(rows))
print('avg ReAct steps:', round(sum(r.react_steps for r in rows) / len(rows), 2))

ReAct correct : 6 / 6
Direct correct: 3 / 6
avg ReAct steps: 2.5


## Recap — what you built

You built a **real ReAct agent** end to end: two safe real tools, the Thought/Action/Observation prompt grammar, the stop-at-observation generation, robust parsing, the reason→act→observe loop with real stop conditions, and an honest ReAct-vs-baseline evaluation — all driving a genuine LLM with reproducible greedy decoding.

The one idea to keep: **an LLM that can act on real observations beats an LLM that can only think.** Reasoning tells the agent *what to do next*; acting + observing tells it *what is actually true*. ReAct interleaves the two, and that interleaving is why it grounds reasoning and cuts hallucination.

Next: [Tool Use & Function Calling](../../03-Tool-Use-and-Function-Calling/03-Tool-Use-and-Function-Calling.md) (the structured-output evolution of the text-parsed actions here), and [Reflection & Self-Critique](../../05-Reflection-and-Self-Critique/05-Reflection-and-Self-Critique.md) (adding a self-correction step between attempts).